In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

INPUT_ROOT = Path(
    "/kaggle/input/datasets/alejandragomezr/results-models-c-tenet"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/comment_12_window_metrics"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "CTE-Net",
    "EEGNet",
    "ShallowConvNet",
    "T-GARNet",
    "IMC-BGT",
    "MultiStream",
]

NAME_MAP = {
    "CTE_Net": "CTE-Net",
    "CTE-Net": "CTE-Net",
    "HybridTransformerTEKTE": "CTE-Net",

    "EEGNet": "EEGNet",

    "Shallow": "ShallowConvNet",
    "ShallowConvNet": "ShallowConvNet",

    "TGARNet": "T-GARNet",
    "T-GARNet": "T-GARNet",

    "IMCBGT": "IMC-BGT",
    "IMC_BGT": "IMC-BGT",
    "IMC-BGT": "IMC-BGT",

    "MultiStream": "MultiStream",
}

METRIC_COLUMNS = [
    "accuracy",
    "sensitivity",
    "specificity",
    "balanced_accuracy",
    "precision",
    "f1_score",
    "roc_auc",
]

METRIC_LABELS = {
    "accuracy": "Accuracy",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "balanced_accuracy": "Balanced accuracy",
    "precision": "Precision",
    "f1_score": "F1-score",
    "roc_auc": "ROC-AUC",
}


# ============================================================
# 2. FUNCIONES
# ============================================================

def normalize_model_name(value):
    value = str(value).strip()
    return NAME_MAP.get(value, value)


def calculate_metrics(y_true, y_prob, threshold=0.5):
    """
    Calcula las métricas por ventana usando ADHD como clase positiva.
    """

    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    if y_true.shape != y_prob.shape:
        raise ValueError(
            "y_true y y_prob deben tener la misma dimensión."
        )

    if not np.isin(y_true, [0, 1]).all():
        raise ValueError(
            "Las etiquetas deben ser binarias: "
            "0=Control y 1=ADHD."
        )

    if not np.isfinite(y_prob).all():
        raise ValueError(
            "Las probabilidades contienen NaN o Inf."
        )

    y_pred = (y_prob >= threshold).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),

        "sensitivity": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),

        "specificity": specificity,

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),

        "f1_score": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),

        "roc_auc": roc_auc,

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def normalize_metric_columns(frame):
    """
    Estandariza distintos nombres posibles de las columnas.
    """

    aliases = {
        "model": "model",
        "model_name": "model",

        "seed": "seed",
        "random_seed": "seed",

        "accuracy": "accuracy",
        "acc": "accuracy",

        "sensitivity": "sensitivity",
        "recall": "sensitivity",
        "sens": "sensitivity",

        "specificity": "specificity",
        "spec": "specificity",

        "balanced_accuracy": "balanced_accuracy",
        "balancedaccuracy": "balanced_accuracy",
        "balanced_acc": "balanced_accuracy",
        "bacc": "balanced_accuracy",

        "precision": "precision",
        "prec": "precision",

        "f1": "f1_score",
        "f1_score": "f1_score",
        "f1score": "f1_score",

        "roc_auc": "roc_auc",
        "rocauc": "roc_auc",
        "auc": "roc_auc",
    }

    rename_columns = {}

    for column in frame.columns:
        normalized = (
            str(column)
            .strip()
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("%", "")
        )

        normalized = "_".join(
            part
            for part in normalized.split("_")
            if part
        )

        if normalized in aliases:
            rename_columns[column] = aliases[normalized]

    return frame.rename(columns=rename_columns)


def infer_model_name_from_path(path):
    """
    Obtiene el modelo a partir del nombre del archivo.
    """

    filename = path.name

    suffixes = [
        "_all_window_predictions.csv",
        "_window_level_metrics_by_seed.csv",
    ]

    inferred_name = filename

    for suffix in suffixes:
        if inferred_name.endswith(suffix):
            inferred_name = inferred_name[:-len(suffix)]
            break

    return normalize_model_name(inferred_name)


def convert_metrics_to_proportions(frame):
    """
    Convierte porcentajes 0--100 al rango 0--1 cuando sea necesario.
    """

    frame = frame.copy()

    for metric in METRIC_COLUMNS:
        frame[metric] = pd.to_numeric(
            frame[metric],
            errors="raise",
        )

        finite_values = frame[metric].dropna()

        if (
            not finite_values.empty
            and finite_values.max() > 1.0001
        ):
            frame[metric] = frame[metric] / 100.0

    return frame


# ============================================================
# 3. LOCALIZAR PREDICCIONES COMPLETAS
# ============================================================

prediction_files = sorted(
    INPUT_ROOT.rglob(
        "*_all_window_predictions.csv"
    )
)

print("=" * 80)
print("PREDICCIONES COMPLETAS ENCONTRADAS")
print("=" * 80)

if prediction_files:
    for path in prediction_files:
        print(" -", path.relative_to(INPUT_ROOT))
else:
    print("No se encontraron archivos de predicciones completas.")


# ============================================================
# 4. CALCULAR MÉTRICAS POR FOLD
# ============================================================

fold_rows = []

for path in prediction_files:
    frame = pd.read_csv(path)

    required_columns = {
        "seed",
        "fold",
        "y_true",
        "prob_adhd",
    }

    missing = required_columns - set(frame.columns)

    if missing:
        raise KeyError(
            f"\n{path.name} no contiene las columnas "
            f"requeridas:\n{sorted(missing)}\n"
            f"Columnas disponibles:\n"
            f"{frame.columns.tolist()}"
        )

    if "model" in frame.columns:
        original_name = str(
            frame["model"].iloc[0]
        )
        model_name = normalize_model_name(
            original_name
        )
    else:
        model_name = infer_model_name_from_path(
            path
        )

    frame["seed"] = frame["seed"].astype(int)
    frame["fold"] = frame["fold"].astype(int)

    for (seed, fold), group in frame.groupby(
        ["seed", "fold"],
        sort=True,
    ):
        metrics = calculate_metrics(
            y_true=group["y_true"].to_numpy(),
            y_prob=group["prob_adhd"].to_numpy(),
            threshold=0.5,
        )

        fold_rows.append({
            "model": model_name,
            "seed": int(seed),
            "fold": int(fold),
            "n_windows": int(len(group)),
            **metrics,
        })

fold_metrics = pd.DataFrame(fold_rows)


# ============================================================
# 5. PROMEDIAR FOLDS DENTRO DE CADA SEMILLA
# ============================================================

if not fold_metrics.empty:
    prediction_seed_metrics = (
        fold_metrics
        .groupby(
            ["model", "seed"],
            as_index=False,
        )[METRIC_COLUMNS]
        .mean()
    )
else:
    prediction_seed_metrics = pd.DataFrame(
        columns=[
            "model",
            "seed",
            *METRIC_COLUMNS,
        ]
    )


# ============================================================
# 6. LOCALIZAR MÉTRICAS YA CALCULADAS
#    EJEMPLO: MULTISTREAM
# ============================================================

precomputed_files = sorted(
    INPUT_ROOT.rglob(
        "*_window_level_metrics_by_seed.csv"
    )
)

print("\n" + "=" * 80)
print("MÉTRICAS POR SEMILLA ENCONTRADAS")
print("=" * 80)

if precomputed_files:
    for path in precomputed_files:
        print(" -", path.relative_to(INPUT_ROOT))
else:
    print("No se encontraron métricas precalculadas.")


# ============================================================
# 7. CARGAR MÉTRICAS PRECALCULADAS
# ============================================================

precomputed_frames = []

models_already_loaded = set(
    prediction_seed_metrics[
        "model"
    ].astype(str)
)

for path in precomputed_files:
    frame = pd.read_csv(path)
    frame = normalize_metric_columns(frame)

    inferred_model = infer_model_name_from_path(
        path
    )

    if "model" not in frame.columns:
        frame["model"] = inferred_model
    else:
        frame["model"] = (
            frame["model"]
            .astype(str)
            .map(normalize_model_name)
        )

    if "seed" not in frame.columns:
        raise KeyError(
            f"\n{path.name} no contiene la columna seed.\n"
            f"Columnas disponibles:\n"
            f"{frame.columns.tolist()}"
        )

    # Balanced accuracy puede obtenerse de sensibilidad y especificidad.
    if (
        "balanced_accuracy" not in frame.columns
        and "sensitivity" in frame.columns
        and "specificity" in frame.columns
    ):
        frame["balanced_accuracy"] = (
            frame["sensitivity"]
            + frame["specificity"]
        ) / 2.0

    missing_metrics = (
        set(METRIC_COLUMNS)
        - set(frame.columns)
    )

    if missing_metrics:
        raise KeyError(
            f"\n{path.name} no contiene las métricas:\n"
            f"{sorted(missing_metrics)}\n"
            f"Columnas disponibles:\n"
            f"{frame.columns.tolist()}"
        )

    frame = frame[
        [
            "model",
            "seed",
            *METRIC_COLUMNS,
        ]
    ].copy()

    frame["seed"] = pd.to_numeric(
        frame["seed"],
        errors="raise",
    ).astype(int)

    frame = convert_metrics_to_proportions(
        frame
    )

    # Si el archivo contiene varias filas por semilla,
    # se promedian para obtener una fila modelo-semilla.
    frame = (
        frame
        .groupby(
            ["model", "seed"],
            as_index=False,
        )[METRIC_COLUMNS]
        .mean()
    )

    for model_name, model_frame in frame.groupby(
        "model",
        sort=False,
    ):
        model_name = normalize_model_name(
            model_name
        )

        model_frame = model_frame.copy()
        model_frame["model"] = model_name

        if model_name in models_already_loaded:
            print(
                "\nArchivo omitido porque el modelo "
                f"{model_name} ya fue cargado:"
            )
            print(" -", path.relative_to(INPUT_ROOT))
            continue

        precomputed_frames.append(
            model_frame
        )
        models_already_loaded.add(
            model_name
        )


# ============================================================
# 8. COMBINAR TODOS LOS MODELOS
# ============================================================

frames_to_combine = []

if not prediction_seed_metrics.empty:
    frames_to_combine.append(
        prediction_seed_metrics
    )

frames_to_combine.extend(
    precomputed_frames
)

if not frames_to_combine:
    raise FileNotFoundError(
        "No fue posible cargar predicciones ni "
        "métricas por semilla."
    )

seed_metrics = pd.concat(
    frames_to_combine,
    ignore_index=True,
)

seed_metrics["model"] = (
    seed_metrics["model"]
    .astype(str)
    .map(normalize_model_name)
)

seed_metrics = seed_metrics.sort_values(
    ["model", "seed"]
).reset_index(drop=True)


# ============================================================
# 9. VALIDACIONES
# ============================================================

duplicated = seed_metrics.duplicated(
    subset=["model", "seed"],
    keep=False,
)

if duplicated.any():
    duplicated_rows = seed_metrics.loc[
        duplicated,
        ["model", "seed"],
    ]

    raise RuntimeError(
        "Existen filas duplicadas para modelo y semilla:\n"
        + duplicated_rows.to_string(index=False)
    )

seed_counts = (
    seed_metrics
    .groupby("model")["seed"]
    .nunique()
)

print("\n" + "=" * 80)
print("SEMILLAS CARGADAS POR MODELO")
print("=" * 80)
print(seed_counts)

models_found = set(
    seed_metrics["model"].astype(str)
)

models_missing = [
    model
    for model in MODEL_ORDER
    if model not in models_found
]

if models_missing:
    print("\nModelos todavía pendientes:")

    for model in models_missing:
        print(" -", model)

if not fold_metrics.empty:
    fold_counts = (
        fold_metrics
        .groupby(
            ["model", "seed"]
        )["fold"]
        .nunique()
    )

    invalid_fold_counts = fold_counts[
        fold_counts != 5
    ]

    if not invalid_fold_counts.empty:
        print(
            "\nADVERTENCIA: algunas semillas "
            "no contienen cinco folds:"
        )
        print(invalid_fold_counts)

invalid_seed_counts = seed_counts[
    seed_counts != 10
]

if not invalid_seed_counts.empty:
    print(
        "\nADVERTENCIA: algunos modelos "
        "no contienen diez semillas:"
    )
    print(invalid_seed_counts)


# ============================================================
# 10. MEDIA Y DESVIACIÓN ESTÁNDAR
#     ENTRE LAS DIEZ SEMILLAS
# ============================================================

summary_rows = []

for model, group in seed_metrics.groupby(
    "model",
    sort=False,
):
    group = group.sort_values("seed")

    for metric in METRIC_COLUMNS:
        values = group[
            metric
        ].to_numpy(dtype=float)

        mean_value = np.nanmean(values)

        std_value = (
            np.nanstd(values, ddof=1)
            if len(values) > 1
            else np.nan
        )

        summary_rows.append({
            "model": model,
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "mean": mean_value,
            "std": std_value,
            "mean_percent": 100.0 * mean_value,
            "std_percent": 100.0 * std_value,
            "n_seeds": int(len(values)),
        })

summary = pd.DataFrame(summary_rows)

summary["Result (%)"] = summary.apply(
    lambda row: (
        f"{row['mean_percent']:.1f} "
        f"$\\pm$ {row['std_percent']:.1f}"
    ),
    axis=1,
)


# ============================================================
# 11. ORDENAR RESULTADOS
# ============================================================

summary["model_order"] = summary[
    "model"
].map({
    model: index
    for index, model in enumerate(MODEL_ORDER)
})

summary["metric_order"] = summary[
    "metric"
].map({
    metric: index
    for index, metric in enumerate(METRIC_COLUMNS)
})

summary = summary.sort_values(
    ["metric_order", "model_order", "model"]
).reset_index(drop=True)

summary = summary.drop(
    columns=[
        "model_order",
        "metric_order",
    ]
)


# ============================================================
# 12. TABLA PARA EL MANUSCRITO
# ============================================================

manuscript_table = summary.pivot(
    index="model",
    columns="metric_label",
    values="Result (%)",
).reset_index()

manuscript_table["model_order"] = (
    manuscript_table["model"].map({
        model: index
        for index, model in enumerate(MODEL_ORDER)
    })
)

manuscript_table = (
    manuscript_table
    .sort_values(
        ["model_order", "model"]
    )
    .drop(columns="model_order")
    .reset_index(drop=True)
)


# ============================================================
# 13. GUARDAR RESULTADOS
# ============================================================

fold_metrics.to_csv(
    OUTPUT_ROOT
    / "window_metrics_by_fold.csv",
    index=False,
)

seed_metrics.to_csv(
    OUTPUT_ROOT
    / "window_metrics_by_seed.csv",
    index=False,
)

summary.to_csv(
    OUTPUT_ROOT
    / "window_metrics_mean_std.csv",
    index=False,
)

manuscript_table.to_csv(
    OUTPUT_ROOT
    / "window_metrics_table_for_manuscript.csv",
    index=False,
)


# ============================================================
# 14. MOSTRAR RESULTADOS
# ============================================================

print("\n" + "=" * 80)
print("RESULTADOS PARA EL COMENTARIO 12")
print("=" * 80)

display(
    summary[
        [
            "metric_label",
            "model",
            "Result (%)",
            "n_seeds",
        ]
    ].rename(
        columns={
            "metric_label": "Metric",
            "model": "Model",
        }
    )
)

print("\nTabla horizontal para el manuscrito:")
display(manuscript_table)

print("\nArchivos guardados en:")
print(OUTPUT_ROOT)

print("\nArchivos generados:")
for path in sorted(OUTPUT_ROOT.glob("*")):
    print(" -", path.name)

PREDICCIONES COMPLETAS ENCONTRADAS
 - C-TENet/HybridTransformerTEKTE_reinference_complete/CTE_Net_all_window_predictions.csv
 - EEGNet/EEGNet_reinference_complete/EEGNet_all_window_predictions.csv
 - IMCBGT_all_window_predictions.csv
 - Shallow/ShallowConvNet_reinference_complete/ShallowConvNet_all_window_predictions.csv
 - T-GARNet/TGARNet_reinference_complete/TGARNet_all_window_predictions.csv

MÉTRICAS POR SEMILLA ENCONTRADAS
 - MultiStream/MultiStream_analysis_only/MultiStream_window_level_metrics_by_seed.csv
 - MultiStream/MultiStream_analysis_only_results/MultiStream_window_level_metrics_by_seed.csv

Archivo omitido porque el modelo MultiStream ya fue cargado:
 - MultiStream/MultiStream_analysis_only_results/MultiStream_window_level_metrics_by_seed.csv

SEMILLAS CARGADAS POR MODELO
model
CTE-Net           10
EEGNet            10
IMC-BGT           10
MultiStream       10
ShallowConvNet    10
T-GARNet          10
Name: seed, dtype: int64

RESULTADOS PARA EL COMENTARIO 12


,Metric,Model,Result (%),n_seeds
0,Accuracy,CTE-Net,80.9 $\pm$ 1.7,10
1,Accuracy,EEGNet,81.5 $\pm$ 2.1,10
2,Accuracy,ShallowConvNet,83.5 $\pm$ 1.7,10
3,Accuracy,T-GARNet,77.6 $\pm$ 0.5,10
4,Accuracy,IMC-BGT,66.3 $\pm$ 1.2,10
5,Accuracy,MultiStream,58.6 $\pm$ 0.3,10
6,Sensitivity,CTE-Net,84.2 $\pm$ 2.3,10
7,Sensitivity,EEGNet,83.5 $\pm$ 3.6,10
8,Sensitivity,ShallowConvNet,80.6 $\pm$ 2.4,10
9,Sensitivity,T-GARNet,85.6 $\pm$ 1.1,10



Tabla horizontal para el manuscrito:


metric_label,model,Accuracy,Balanced accuracy,F1-score,Precision,ROC-AUC,Sensitivity,Specificity
0,CTE-Net,80.9 $\pm$ 1.7,80.6 $\pm$ 1.8,82.9 $\pm$ 1.5,82.7 $\pm$ 2.1,87.9 $\pm$ 1.5,84.2 $\pm$ 2.3,77.0 $\pm$ 3.2
1,EEGNet,81.5 $\pm$ 2.1,81.4 $\pm$ 2.1,83.1 $\pm$ 2.2,84.3 $\pm$ 2.2,88.8 $\pm$ 2.5,83.5 $\pm$ 3.6,79.3 $\pm$ 4.0
2,ShallowConvNet,83.5 $\pm$ 1.7,83.7 $\pm$ 1.8,83.2 $\pm$ 2.0,88.1 $\pm$ 2.9,90.4 $\pm$ 1.7,80.6 $\pm$ 2.4,86.9 $\pm$ 3.4
3,T-GARNet,77.6 $\pm$ 0.5,76.7 $\pm$ 0.6,80.8 $\pm$ 0.4,77.3 $\pm$ 0.8,84.0 $\pm$ 0.4,85.6 $\pm$ 1.1,67.8 $\pm$ 1.9
4,IMC-BGT,66.3 $\pm$ 1.2,65.3 $\pm$ 1.4,70.8 $\pm$ 1.0,68.2 $\pm$ 1.6,71.2 $\pm$ 1.0,74.7 $\pm$ 2.7,55.8 $\pm$ 4.5
5,MultiStream,58.6 $\pm$ 0.3,55.2 $\pm$ 0.4,69.8 $\pm$ 0.3,58.7 $\pm$ 0.3,55.3 $\pm$ 0.4,86.1 $\pm$ 0.9,24.4 $\pm$ 1.4



Archivos guardados en:
/kaggle/working/comment_12_window_metrics

Archivos generados:
 - window_metrics_by_fold.csv
 - window_metrics_by_seed.csv
 - window_metrics_mean_std.csv
 - window_metrics_table_for_manuscript.csv


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

INPUT_PATH = Path(
    "/kaggle/working/comment_12_window_metrics/"
    "window_metrics_by_seed.csv"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/comment_12_wilcoxon_holm"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REFERENCE_MODEL = "CTE-Net"

BASELINE_ORDER = [
    "EEGNet",
    "ShallowConvNet",
    "T-GARNet",
    "IMC-BGT",
    "MultiStream",
]

METRICS = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "sensitivity": "Sensitivity",
}

ALPHA = 0.05

NAME_MAP = {
    "CTE_Net": "CTE-Net",
    "CTE-Net": "CTE-Net",
    "HybridTransformerTEKTE": "CTE-Net",

    "EEGNet": "EEGNet",

    "Shallow": "ShallowConvNet",
    "ShallowConvNet": "ShallowConvNet",

    "TGARNet": "T-GARNet",
    "T-GARNet": "T-GARNet",

    "IMCBGT": "IMC-BGT",
    "IMC_BGT": "IMC-BGT",
    "IMC-BGT": "IMC-BGT",

    "MultiStream": "MultiStream",
}


# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def normalize_model_name(value):
    value = str(value).strip()
    return NAME_MAP.get(value, value)


def format_result(mean_value, std_value):
    return (
        f"{100.0 * mean_value:.1f} "
        f"$\\pm$ {100.0 * std_value:.1f}"
    )


def format_p_latex(p_value):
    if p_value < 0.0001:
        exponent = int(np.floor(np.log10(p_value)))
        coefficient = p_value / (10 ** exponent)

        return (
            f"${coefficient:.2f}"
            f"\\times10^{{{exponent}}}$"
        )

    return f"{p_value:.4f}"


# ============================================================
# 3. CARGAR MÉTRICAS POR SEMILLA
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{INPUT_PATH}"
    )

seed_metrics = pd.read_csv(INPUT_PATH)

required_columns = {
    "model",
    "seed",
    *METRICS.keys(),
}

missing_columns = (
    required_columns
    - set(seed_metrics.columns)
)

if missing_columns:
    raise KeyError(
        "Faltan las columnas:\n"
        f"{sorted(missing_columns)}\n\n"
        "Columnas disponibles:\n"
        f"{seed_metrics.columns.tolist()}"
    )

seed_metrics = seed_metrics.copy()

seed_metrics["model"] = (
    seed_metrics["model"]
    .astype(str)
    .map(normalize_model_name)
)

seed_metrics["seed"] = pd.to_numeric(
    seed_metrics["seed"],
    errors="raise",
).astype(int)

for metric in METRICS:
    seed_metrics[metric] = pd.to_numeric(
        seed_metrics[metric],
        errors="raise",
    )

    # Convertir porcentajes 0--100 a proporciones 0--1.
    if seed_metrics[metric].max() > 1.0001:
        seed_metrics[metric] /= 100.0


# ============================================================
# 4. VALIDAR UNA FILA POR MODELO Y SEMILLA
# ============================================================

duplicated = seed_metrics.duplicated(
    subset=["model", "seed"],
    keep=False,
)

if duplicated.any():
    raise RuntimeError(
        "Existen filas duplicadas para modelo y semilla:\n"
        + seed_metrics.loc[
            duplicated,
            ["model", "seed"],
        ].to_string(index=False)
    )

seed_counts = (
    seed_metrics
    .groupby("model")["seed"]
    .nunique()
)

print("=" * 80)
print("SEMILLAS POR MODELO")
print("=" * 80)
print(seed_counts)

invalid_counts = seed_counts[
    seed_counts != 10
]

if not invalid_counts.empty:
    raise RuntimeError(
        "Cada modelo debe contener exactamente diez semillas:\n"
        + invalid_counts.to_string()
    )

if REFERENCE_MODEL not in set(
    seed_metrics["model"]
):
    raise RuntimeError(
        "No se encontraron resultados para CTE-Net."
    )


# ============================================================
# 5. IDENTIFICAR BASELINES DISPONIBLES
# ============================================================

available_models = set(
    seed_metrics["model"]
)

available_baselines = [
    model
    for model in BASELINE_ORDER
    if model in available_models
]

missing_baselines = [
    model
    for model in BASELINE_ORDER
    if model not in available_models
]

print("\nBaselines disponibles:")
for model in available_baselines:
    print(" -", model)

if missing_baselines:
    print("\nBaselines pendientes:")
    for model in missing_baselines:
        print(" -", model)


# ============================================================
# 6. WILCOXON PAREADO
# ============================================================

comparison_rows = []

for metric, metric_label in METRICS.items():

    reference = (
        seed_metrics.loc[
            seed_metrics["model"] == REFERENCE_MODEL,
            ["seed", metric],
        ]
        .rename(
            columns={
                metric: "reference_value"
            }
        )
    )

    for baseline in available_baselines:

        comparison = (
            seed_metrics.loc[
                seed_metrics["model"] == baseline,
                ["seed", metric],
            ]
            .rename(
                columns={
                    metric: "baseline_value"
                }
            )
        )

        paired = reference.merge(
            comparison,
            on="seed",
            how="inner",
            validate="one_to_one",
        ).sort_values("seed")

        if len(paired) != 10:
            raise RuntimeError(
                f"{metric_label}: CTE-Net vs. {baseline} "
                f"solo tiene {len(paired)} semillas pareadas."
            )

        reference_values = paired[
            "reference_value"
        ].to_numpy(dtype=float)

        baseline_values = paired[
            "baseline_value"
        ].to_numpy(dtype=float)

        differences = (
            reference_values
            - baseline_values
        )

        if np.allclose(differences, 0.0):
            statistic = 0.0
            p_raw = 1.0
        else:
            test_result = wilcoxon(
                reference_values,
                baseline_values,
                zero_method="wilcox",
                correction=False,
                alternative="two-sided",
                method="auto",
            )

            statistic = float(
                test_result.statistic
            )
            p_raw = float(
                test_result.pvalue
            )

        reference_mean = float(
            np.mean(reference_values)
        )
        reference_std = float(
            np.std(reference_values, ddof=1)
        )

        baseline_mean = float(
            np.mean(baseline_values)
        )
        baseline_std = float(
            np.std(baseline_values, ddof=1)
        )

        comparison_rows.append({
            "metric": metric,
            "metric_label": metric_label,
            "baseline": baseline,
            "n_seeds": int(len(paired)),

            "cte_mean": reference_mean,
            "cte_std": reference_std,

            "baseline_mean": baseline_mean,
            "baseline_std": baseline_std,

            "mean_difference_cte_minus_baseline": (
                reference_mean
                - baseline_mean
            ),

            "wilcoxon_statistic": statistic,
            "p_raw": p_raw,
        })

results = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 7. CORRECCIÓN DE HOLM
# ============================================================

reject, p_holm, _, _ = multipletests(
    results["p_raw"].to_numpy(),
    alpha=ALPHA,
    method="holm",
)

results["p_holm"] = p_holm
results["significant_holm"] = reject


# ============================================================
# 8. DIRECCIÓN DE LA COMPARACIÓN
# ============================================================

def determine_comparison(row):
    if not row["significant_holm"]:
        return (
            f"CTE-Net $\\approx$ "
            f"{row['baseline']}"
        )

    if (
        row[
            "mean_difference_cte_minus_baseline"
        ] > 0
    ):
        return (
            f"CTE-Net $>$ "
            f"{row['baseline']}"
        )

    return (
        f"CTE-Net $<$ "
        f"{row['baseline']}"
    )


results["comparison"] = results.apply(
    determine_comparison,
    axis=1,
)

results["baseline_result_latex"] = (
    results.apply(
        lambda row: format_result(
            row["baseline_mean"],
            row["baseline_std"],
        ),
        axis=1,
    )
)

results["cte_result_latex"] = (
    results.apply(
        lambda row: format_result(
            row["cte_mean"],
            row["cte_std"],
        ),
        axis=1,
    )
)

results["p_raw_latex"] = (
    results["p_raw"]
    .map(format_p_latex)
)

results["p_holm_latex"] = (
    results["p_holm"]
    .map(format_p_latex)
)


# ============================================================
# 9. ORDENAR RESULTADOS
# ============================================================

metric_order = {
    metric: index
    for index, metric in enumerate(
        METRICS.keys()
    )
}

baseline_order = {
    model: index
    for index, model in enumerate(
        BASELINE_ORDER
    )
}

results["metric_order"] = (
    results["metric"]
    .map(metric_order)
)

results["baseline_order"] = (
    results["baseline"]
    .map(baseline_order)
)

results = (
    results
    .sort_values(
        ["metric_order", "baseline_order"]
    )
    .drop(
        columns=[
            "metric_order",
            "baseline_order",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 10. VALIDAR LA FAMILIA DE COMPARACIONES
# ============================================================

expected_comparisons = (
    len(METRICS)
    * len(BASELINE_ORDER)
)

observed_comparisons = len(results)

final_family_complete = (
    observed_comparisons
    == expected_comparisons
)

print("\n" + "=" * 80)
print("CORRECCIÓN DE HOLM")
print("=" * 80)

print(
    "Comparaciones observadas:",
    observed_comparisons,
)

print(
    "Comparaciones finales esperadas:",
    expected_comparisons,
)

if final_family_complete:
    print(
        "La familia está completa: "
        "3 métricas × 5 baselines = 15 pruebas."
    )
else:
    print(
        "\nADVERTENCIA: la corrección de Holm "
        "todavía es provisional."
    )
    print(
        "Debe ejecutarse nuevamente cuando "
        "estén disponibles todos los baselines."
    )


# ============================================================
# 11. TABLA RESUMIDA
# ============================================================

display_columns = [
    "metric_label",
    "baseline",
    "baseline_result_latex",
    "cte_result_latex",
    "wilcoxon_statistic",
    "p_raw",
    "p_holm",
    "significant_holm",
    "comparison",
]

print("\n" + "=" * 80)
print("WILCOXON PAREADO CON CORRECCIÓN DE HOLM")
print("=" * 80)

display(
    results[display_columns].rename(
        columns={
            "metric_label": "Metric",
            "baseline": "Baseline",
            "baseline_result_latex": "Baseline result (%)",
            "cte_result_latex": "CTE-Net result (%)",
            "wilcoxon_statistic": "Wilcoxon statistic",
            "p_raw": "Raw p",
            "p_holm": "Holm p",
            "significant_holm": "Significant",
            "comparison": "Comparison",
        }
    )
)


# ============================================================
# 12. GUARDAR RESULTADOS
# ============================================================

results.to_csv(
    OUTPUT_ROOT
    / "CTE_Net_vs_baselines_Wilcoxon_Holm_updated.csv",
    index=False,
)

manuscript_rows = results[
    [
        "metric_label",
        "baseline",
        "baseline_result_latex",
        "p_holm_latex",
        "comparison",
    ]
].copy()

manuscript_rows.to_csv(
    OUTPUT_ROOT
    / "CTE_Net_vs_baselines_Wilcoxon_Holm_manuscript.csv",
    index=False,
)

print("\nArchivos guardados en:")
print(OUTPUT_ROOT)

for path in sorted(
    OUTPUT_ROOT.glob("*")
):
    print(" -", path.name)

SEMILLAS POR MODELO
model
CTE-Net           10
EEGNet            10
IMC-BGT           10
MultiStream       10
ShallowConvNet    10
T-GARNet          10
Name: seed, dtype: int64

Baselines disponibles:
 - EEGNet
 - ShallowConvNet
 - T-GARNet
 - IMC-BGT
 - MultiStream

CORRECCIÓN DE HOLM
Comparaciones observadas: 15
Comparaciones finales esperadas: 15
La familia está completa: 3 métricas × 5 baselines = 15 pruebas.

WILCOXON PAREADO CON CORRECCIÓN DE HOLM


,Metric,Baseline,Baseline result (%),CTE-Net result (%),Wilcoxon statistic,Raw p,Holm p,Significant,Comparison
0,Accuracy,EEGNet,81.5 $\pm$ 2.1,80.9 $\pm$ 1.7,22.0,0.625000,1.000000,False,CTE-Net $\approx$ EEGNet
1,Accuracy,ShallowConvNet,83.5 $\pm$ 1.7,80.9 $\pm$ 1.7,3.0,0.009766,0.068359,False,CTE-Net $\approx$ ShallowConvNet
2,Accuracy,T-GARNet,77.6 $\pm$ 0.5,80.9 $\pm$ 1.7,0.0,0.001953,0.029297,True,CTE-Net $>$ T-GARNet
3,Accuracy,IMC-BGT,66.3 $\pm$ 1.2,80.9 $\pm$ 1.7,0.0,0.001953,0.029297,True,CTE-Net $>$ IMC-BGT
4,Accuracy,MultiStream,58.6 $\pm$ 0.3,80.9 $\pm$ 1.7,0.0,0.001953,0.029297,True,CTE-Net $>$ MultiStream
5,Precision,EEGNet,84.3 $\pm$ 2.2,82.7 $\pm$ 2.1,14.0,0.193359,0.773438,False,CTE-Net $\approx$ EEGNet
6,Precision,ShallowConvNet,88.1 $\pm$ 2.9,82.7 $\pm$ 2.1,0.0,0.001953,0.029297,True,CTE-Net $<$ ShallowConvNet
7,Precision,T-GARNet,77.3 $\pm$ 0.8,82.7 $\pm$ 2.1,0.0,0.001953,0.029297,True,CTE-Net $>$ T-GARNet
8,Precision,IMC-BGT,68.2 $\pm$ 1.6,82.7 $\pm$ 2.1,0.0,0.001953,0.029297,True,CTE-Net $>$ IMC-BGT
9,Precision,MultiStream,58.7 $\pm$ 0.3,82.7 $\pm$ 2.1,0.0,0.001953,0.029297,True,CTE-Net $>$ MultiStream



Archivos guardados en:
/kaggle/working/comment_12_wilcoxon_holm
 - CTE_Net_vs_baselines_Wilcoxon_Holm_manuscript.csv
 - CTE_Net_vs_baselines_Wilcoxon_Holm_updated.csv


In [3]:
# ============================================================
# MATERIAL SUPLEMENTARIO: MATRICES DE CONFUSIÓN NORMALIZADAS
# ============================================================
# Cada matriz se normaliza por la clase verdadera. Dentro de cada
# semilla, sensibilidad y especificidad ya corresponden a la media
# de los cinco folds. Después se calculan la media y la desviación
# estándar muestral entre las diez semillas.

SUPPLEMENTARY_OUTPUT_ROOT = Path(
    "/kaggle/working/comment_12_supplementary"
)
SUPPLEMENTARY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUPPLEMENTARY_CSV = (
    SUPPLEMENTARY_OUTPUT_ROOT
    / "Supplementary_Data_S1_window_level_normalized_confusion_matrices.csv"
)

required_columns = {
    "model",
    "seed",
    "sensitivity",
    "specificity",
}
missing_columns = required_columns - set(seed_metrics.columns)

if missing_columns:
    raise KeyError(
        "No es posible construir las matrices de confusión. "
        f"Faltan las columnas: {sorted(missing_columns)}"
    )

confusion_source = seed_metrics[
    ["model", "seed", "sensitivity", "specificity"]
].copy()

confusion_source["model"] = (
    confusion_source["model"]
    .astype(str)
    .map(normalize_model_name)
)
confusion_source["seed"] = pd.to_numeric(
    confusion_source["seed"], errors="raise"
).astype(int)

for metric in ["sensitivity", "specificity"]:
    confusion_source[metric] = pd.to_numeric(
        confusion_source[metric], errors="raise"
    )

    if confusion_source[metric].max() > 1.0001:
        confusion_source[metric] /= 100.0

    if not confusion_source[metric].between(0.0, 1.0).all():
        raise ValueError(
            f"{metric} contiene valores fuera del intervalo [0, 1]."
        )

duplicate_seed_rows = confusion_source.duplicated(
    ["model", "seed"], keep=False
)
if duplicate_seed_rows.any():
    raise RuntimeError(
        "Se encontraron filas duplicadas para una combinación modelo-semilla."
    )

confusion_seed_rows = []

for _, row in confusion_source.iterrows():
    specificity = float(row["specificity"])
    sensitivity = float(row["sensitivity"])

    matrix_cells = [
        ("Control", "Control", "TN", specificity),
        ("Control", "ADHD", "FP", 1.0 - specificity),
        ("ADHD", "Control", "FN", 1.0 - sensitivity),
        ("ADHD", "ADHD", "TP", sensitivity),
    ]

    for true_class, predicted_class, matrix_cell, value in matrix_cells:
        confusion_seed_rows.append({
            "model": row["model"],
            "seed": int(row["seed"]),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "matrix_cell": matrix_cell,
            "normalized_value": float(value),
        })

confusion_by_seed = pd.DataFrame(confusion_seed_rows)

confusion_summary = (
    confusion_by_seed
    .groupby(
        [
            "model",
            "true_class",
            "predicted_class",
            "matrix_cell",
        ],
        as_index=False,
    )["normalized_value"]
    .agg(["mean", "std", "count"])
    .rename(columns={"count": "n_seeds"})
)

confusion_summary["mean_percent"] = (
    100.0 * confusion_summary["mean"]
)
confusion_summary["std_percent"] = (
    100.0 * confusion_summary["std"]
)
confusion_summary["normalization"] = "row-normalized by true class"
confusion_summary["aggregation"] = (
    "five-fold mean within seed, followed by mean and sample SD across seeds"
)
confusion_summary["positive_class"] = "ADHD"
confusion_summary["decision_threshold"] = 0.5

model_order_map = {
    model: index for index, model in enumerate(MODEL_ORDER)
}
true_class_order = {"Control": 0, "ADHD": 1}
predicted_class_order = {"Control": 0, "ADHD": 1}

confusion_summary["_model_order"] = (
    confusion_summary["model"].map(model_order_map)
)
confusion_summary["_true_order"] = (
    confusion_summary["true_class"].map(true_class_order)
)
confusion_summary["_predicted_order"] = (
    confusion_summary["predicted_class"].map(predicted_class_order)
)

confusion_summary = (
    confusion_summary
    .sort_values(
        ["_model_order", "_true_order", "_predicted_order"]
    )
    .drop(columns=["_model_order", "_true_order", "_predicted_order"])
    .reset_index(drop=True)
)

# Cada fila de una matriz normalizada debe sumar 1.
row_sums = (
    confusion_summary
    .groupby(["model", "true_class"])["mean"]
    .sum()
)
if not np.allclose(row_sums.to_numpy(), 1.0, atol=1e-10):
    raise RuntimeError(
        "Las filas de alguna matriz normalizada no suman 1."
    )

if not (confusion_summary["n_seeds"] == 10).all():
    raise RuntimeError(
        "Alguna celda de las matrices no contiene diez semillas."
    )

confusion_summary.to_csv(SUPPLEMENTARY_CSV, index=False)

print("Archivo suplementario guardado en:")
print(SUPPLEMENTARY_CSV)
display(
    confusion_summary[
        [
            "model",
            "true_class",
            "predicted_class",
            "matrix_cell",
            "mean_percent",
            "std_percent",
            "n_seeds",
        ]
    ]
)

Archivo suplementario guardado en:
/kaggle/working/comment_12_supplementary/Supplementary_Data_S1_window_level_normalized_confusion_matrices.csv


,model,true_class,predicted_class,matrix_cell,mean_percent,std_percent,n_seeds
0,CTE-Net,Control,Control,TN,76.970474,3.230479,10
1,CTE-Net,Control,ADHD,FP,23.029526,3.230479,10
2,CTE-Net,ADHD,Control,FN,15.813121,2.316226,10
3,CTE-Net,ADHD,ADHD,TP,84.186879,2.316226,10
4,EEGNet,Control,Control,TN,79.283708,3.988703,10
5,EEGNet,Control,ADHD,FP,20.716292,3.988703,10
6,EEGNet,ADHD,Control,FN,16.507954,3.573241,10
7,EEGNet,ADHD,ADHD,TP,83.492046,3.573241,10
8,ShallowConvNet,Control,Control,TN,86.912857,3.444741,10
9,ShallowConvNet,Control,ADHD,FP,13.087143,3.444741,10
